# 05 — Image Quality

## Objective

Measure exploratory brightness, contrast, exposure, and sharpness characteristics for complete images and coin regions of interest.

## Motivation

Extreme visual characteristics may reveal samples that deserve inspection. Robust median/MAD outlier detection prioritizes review without automatically classifying an image as defective.

## Inputs

- `curation/outputs/01-dataset-audit/inventory.csv`
- `curation/outputs/02-annotation-audit/annotations_normalized.csv`
- Images from the configured image directory.

## Outputs

- `curation/outputs/05-image-quality/image_quality_metrics.csv`
- `curation/outputs/05-image-quality/image_quality_outliers.csv`
- `curation/outputs/05-image-quality/manifest.yaml`

Per-ROI intermediate metrics are not persisted.


In [ ]:
# Environment-specific setup
import pathlib
import sys

if 'google.colab' not in str(get_ipython()):
    base_folder = pathlib.Path('../../../')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')

ROOT = base_folder.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
from curation.common import load_config, prepare_dataset, stage_output_dir

CONFIG = load_config(ROOT)
DATASET = prepare_dataset(ROOT)
IMAGES_DIR = DATASET["images_dir"]
ANNOTATIONS_DIR = DATASET["annotations_dir"]

from curation.common import crop_bbox, write_manifest
import numpy as np
import pandas as pd
from PIL import Image
import cv2

STAGE = "05-image-quality"
STAGE_DIR = stage_output_dir(STAGE, ROOT)
INVENTORY_PATH = stage_output_dir("01-dataset-audit", ROOT, create=False) / "inventory.csv"
ANNOTATIONS_PATH = stage_output_dir("02-annotation-audit", ROOT, create=False) / "annotations_normalized.csv"
if not INVENTORY_PATH.is_file() or not ANNOTATIONS_PATH.is_file():
    raise FileNotFoundError("Run notebooks 01 and 02 first.")

In [ ]:
quality_config = CONFIG["audits"]["image_quality"]
dark_max = int(quality_config["dark_pixel_max"])
bright_min = int(quality_config["bright_pixel_min"])

def quality_metrics(gray):
    pixels = np.asarray(gray, dtype=np.uint8)
    if not pixels.size:
        return {key: np.nan for key in ("brightness_mean", "contrast_std", "dark_fraction", "bright_fraction", "laplacian_variance")}
    return {
        "brightness_mean": float(pixels.mean()),
        "contrast_std": float(pixels.std()),
        "dark_fraction": float((pixels <= dark_max).mean()),
        "bright_fraction": float((pixels >= bright_min).mean()),
        "laplacian_variance": float(cv2.Laplacian(pixels, cv2.CV_64F).var()),
    }

inventory = pd.read_csv(INVENTORY_PATH)
annotations = pd.read_csv(ANNOTATIONS_PATH, dtype={"label": str})
coins = annotations[annotations["label"].isin(CONFIG["classes"]["coins"])].dropna(subset=["bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax"])

full_rows = []
for row in inventory[inventory["readable"]].itertuples(index=False):
    with Image.open(IMAGES_DIR / row.relative_path).convert("L") as image:
        metrics = quality_metrics(image)
    full_rows.append({"relative_path": row.relative_path, **{f"full_{key}": value for key, value in metrics.items()}})

roi_rows = []
padding = float(quality_config["crop_padding_fraction"])
for _, row in coins.iterrows():
    with Image.open(IMAGES_DIR / row["relative_path"]).convert("L") as image:
        crop = crop_bbox(image, row, padding)
    if crop is not None:
        metrics = quality_metrics(crop)
        roi_rows.append({"annotation_id": row["annotation_id"], "relative_path": row["relative_path"], **{f"roi_{key}": value for key, value in metrics.items()}})

full_quality = pd.DataFrame(full_rows)
roi_quality = pd.DataFrame(roi_rows)
roi_aggregate = roi_quality.groupby("relative_path").agg(
    roi_coin_count=("annotation_id", "count"),
    roi_brightness_median=("roi_brightness_mean", "median"),
    roi_contrast_median=("roi_contrast_std", "median"),
    roi_dark_fraction_median=("roi_dark_fraction", "median"),
    roi_bright_fraction_median=("roi_bright_fraction", "median"),
    roi_laplacian_variance_median=("roi_laplacian_variance", "median"),
).reset_index()
image_quality = full_quality.merge(roi_aggregate, on="relative_path", how="left")
metrics_path = STAGE_DIR / "image_quality_metrics.csv"
image_quality.to_csv(metrics_path, index=False)

In [ ]:
threshold = float(quality_config["robust_z_threshold"])
def robust_z(series, log1p=False):
    values = pd.to_numeric(series, errors="coerce").astype(float)
    transformed = np.log1p(values.clip(lower=0)) if log1p else values
    median = transformed.median()
    mad = (transformed - median).abs().median()
    return pd.Series(np.nan, index=series.index) if not np.isfinite(mad) or mad == 0 else 0.6744897501960817 * (transformed - median) / mad

rules = [
    ("full_brightness_mean", "both", False), ("full_contrast_std", "low", False),
    ("full_dark_fraction", "high", False), ("full_bright_fraction", "high", False),
    ("full_laplacian_variance", "low", True), ("roi_brightness_median", "both", False),
    ("roi_contrast_median", "low", False), ("roi_dark_fraction_median", "high", False),
    ("roi_bright_fraction_median", "high", False), ("roi_laplacian_variance_median", "low", True),
]
outlier_rows = []
for metric, direction, log1p in rules:
    scores = robust_z(image_quality[metric], log1p)
    mask = scores.abs() >= threshold if direction == "both" else scores <= -threshold if direction == "low" else scores >= threshold
    for index in image_quality.index[mask.fillna(False)]:
        outlier_rows.append({"relative_path": image_quality.at[index, "relative_path"], "metric": metric, "metric_value": image_quality.at[index, metric], "robust_z": float(scores.at[index]), "direction": direction, "status": "statistical_outlier_for_review"})

outliers = pd.DataFrame(outlier_rows, columns=["relative_path", "metric", "metric_value", "robust_z", "direction", "status"])
if len(outliers):
    outliers = outliers.sort_values(["metric", "robust_z"], key=lambda values: values.abs() if values.name == "robust_z" else values, ascending=[True, False])
outliers_path = STAGE_DIR / "image_quality_outliers.csv"
outliers.to_csv(outliers_path, index=False)
display(outliers.head(30))

In [ ]:
write_manifest(
    STAGE,
    "05-image-quality.ipynb",
    inputs={"inventory": INVENTORY_PATH, "annotations_normalized": ANNOTATIONS_PATH},
    parameters=dict(quality_config),
    artifacts=[metrics_path, outliers_path],
    summary={"images_measured": int(len(image_quality)), "coin_rois_measured": int(len(roi_quality)), "outlier_rows": int(len(outliers)), "images_flagged": int(outliers["relative_path"].nunique()) if len(outliers) else 0},
    repo_root=ROOT,
)

## Inspect an Image-Quality Outlier

Use the zero-based row index from `image_quality_outliers.csv` to display the corresponding image. The function also prints the complete CSV row and its metric information.

In [ ]:
def show_quality_outlier(csv_index):
    """Display the image-quality outlier at a CSV row.

    Args:
        csv_index: Zero-based row index in image_quality_outliers.csv.
    """
    import matplotlib.pyplot as plt

    rows = pd.read_csv(outliers_path)
    if not isinstance(csv_index, (int, np.integer)):
        raise TypeError("csv_index must be an integer.")
    if csv_index < 0 or csv_index >= len(rows):
        raise IndexError(f"csv_index must be between 0 and {len(rows) - 1}.")

    outlier = rows.iloc[int(csv_index)]
    image_path = IMAGES_DIR / outlier["relative_path"]

    with Image.open(image_path).convert("RGB") as image:
        plt.figure(figsize=(9, 6))
        plt.imshow(image)
    plt.title(
        f"CSV row {csv_index} | {outlier['metric']} | "
        f"robust z={outlier['robust_z']:.3f}"
    )
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    print(outlier.to_string())
    return outlier

# Example:
show_quality_outlier(0)

In [ ]:
for i in range(10):
    show_quality_outlier(i)

In [ ]:
show_quality_outlier(1847)